# VisionAssist Phase 8 — Colab-resilient QLoRA infrastructure

This notebook runs the Phase 8 sequence:

1. clone/pull the repository;
2. restore the prepared-data archive from Drive;
3. install the training environment;
4. inspect the GPU;
5. normalize legacy Windows image paths in the restored JSONL files;
6. configure persistent Drive checkpoints;
7. run one-batch validation;
8. run the 32-example overfit experiment;
9. interrupt/restart safely and resume from the newest checkpoint;
10. retain only the newest two checkpoints plus the best checkpoint.

Select **Runtime → Change runtime type → GPU** before running.

This revision applies an A100-40GB-safe profile: 2,048 total tokens, 100,352–200,704 image pixels, LoRA rank 8, and attention-only LoRA targets. The smoke test performs both forward and backward passes and records peak VRAM.


In [1]:
#@title 1. Settings and CUDA allocator
import os
from pathlib import Path

# Must be set before the first CUDA/PyTorch initialization.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = (
    "expandable_segments:True,garbage_collection_threshold:0.8"
)
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/moshiur00/visionassist-industrial-visual-inspection.git"
REPO_BRANCH = "main"

DRIVE_ROOT = Path("/content/drive/MyDrive/visionassist")
DRIVE_DATA_ARCHIVE = DRIVE_ROOT / "data/visionassist_prepared_data.tar.gz"
PROJECT_ROOT = Path("/content/visionassist-industrial-visual-inspection")

OVERFIT_CONFIG = PROJECT_ROOT / "configs/training/qwen25vl3b_qlora_overfit.yaml"
OVERFIT_RUN_ID = "qwen25vl3b_qlora_overfit_v1"
DRIVE_CHECKPOINT_ROOT = DRIVE_ROOT / "checkpoints" / OVERFIT_RUN_ID

assert "<YOUR_USERNAME>" not in REPO_URL, "Set REPO_URL first."

## Important runtime rule

Run this notebook from a fresh GPU session. The allocator setting in Cell 1 must be applied before CUDA is initialized. If you already ran an older notebook in the same session, choose **Runtime → Restart session**, then run this notebook from the top.


In [2]:
#@title 2. Mount Drive
from google.colab import drive
drive.mount("/content/drive")
for path in [DRIVE_ROOT/"data", DRIVE_ROOT/"checkpoints", DRIVE_ROOT/"outputs"]:
    path.mkdir(parents=True, exist_ok=True)
assert DRIVE_DATA_ARCHIVE.is_file(), f"Missing: {DRIVE_DATA_ARCHIVE}"

Mounted at /content/drive


In [3]:
#@title 3. Verify GPU
import shutil, subprocess, torch
assert torch.cuda.is_available(), "Choose a GPU runtime."
props = torch.cuda.get_device_properties(0)
print("CUDA allocator:", os.environ.get("PYTORCH_CUDA_ALLOC_CONF"))
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GiB:", round(props.total_memory/1024**3, 2))
print("BF16:", torch.cuda.is_bf16_supported())
print("Free disk GiB:", round(shutil.disk_usage('/content').free/1024**3, 2))
subprocess.run(["nvidia-smi"], check=False)

CUDA allocator: expandable_segments:True,garbage_collection_threshold:0.8
GPU: NVIDIA A100-SXM4-40GB
VRAM GiB: 39.49
BF16: True
Free disk GiB: 188.94


CompletedProcess(args=['nvidia-smi'], returncode=0)

In [4]:
#@title 4. Install uv and clone/pull repository
import subprocess, sys, shutil
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "uv"], check=True)
if (PROJECT_ROOT/".git").is_dir():
    subprocess.run(["git","fetch","origin",REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git","checkout",REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
    subprocess.run(["git","pull","--ff-only","origin",REPO_BRANCH], cwd=PROJECT_ROOT, check=True)
else:
    if PROJECT_ROOT.exists(): shutil.rmtree(PROJECT_ROOT)
    subprocess.run(["git","clone","--branch",REPO_BRANCH,REPO_URL,str(PROJECT_ROOT)], check=True)
print(subprocess.check_output(["git","rev-parse","HEAD"], cwd=PROJECT_ROOT, text=True).strip())

52d6a372a36b86e33d304a8613f8b9f4be57dbb7


In [5]:
#@title 5. Install dependencies
import os, subprocess
os.chdir(PROJECT_ROOT)
subprocess.run(["uv","sync","--extra","training","--extra","dev"], check=True)

CompletedProcess(args=['uv', 'sync', '--extra', 'training', '--extra', 'dev'], returncode=0)

In [6]:
#@title 6. Restore prepared data to fast local storage
import tarfile
required = [
    PROJECT_ROOT/"data/raw/visa",
    PROJECT_ROOT/"data/processed/visa_instructions/train.jsonl",
    PROJECT_ROOT/"data/processed/visa_instructions/validation.jsonl",
]
if not all(path.exists() for path in required):
    with tarfile.open(DRIVE_DATA_ARCHIVE, "r:gz") as archive:
        archive.extractall(PROJECT_ROOT)
assert all(path.exists() for path in required)
print("Prepared data ready.")

/tmp/ipykernel_2204/999844528.py:10: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  archive.extractall(PROJECT_ROOT)


Prepared data ready.


In [7]:
#@title 7. Normalize legacy Windows image paths in instruction JSONL
import json
from pathlib import Path, PurePosixPath

MARKER = ("data", "raw", "visa")


def to_project_relative_image_path(value: str) -> str:
    normalized = value.replace("\\", "/")
    parts = PurePosixPath(normalized).parts
    lowered = tuple(part.lower() for part in parts)

    for index in range(len(parts) - len(MARKER) + 1):
        if lowered[index : index + len(MARKER)] == MARKER:
            return PurePosixPath(*parts[index:]).as_posix()

    candidate = PurePosixPath(normalized)
    if not candidate.is_absolute() and ".." not in candidate.parts:
        return candidate.as_posix()

    raise ValueError(f"Cannot normalize image path: {value}")


def normalize_instruction_jsonl(path: Path) -> tuple[int, int]:
    records: list[dict] = []
    changed_records = 0
    changed_paths = 0

    with path.open("r", encoding="utf-8") as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue

            record = json.loads(line)
            record_changed = False

            for message in record.get("messages", []):
                if message.get("role") != "user":
                    continue

                for item in message.get("content", []):
                    if item.get("type") != "image":
                        continue

                    original = str(item.get("image", ""))
                    normalized = to_project_relative_image_path(original)

                    if normalized != original:
                        item["image"] = normalized
                        changed_paths += 1
                        record_changed = True

            if record_changed:
                changed_records += 1

            records.append(record)

    temporary = path.with_suffix(path.suffix + ".tmp")
    with temporary.open("w", encoding="utf-8", newline="\n") as handle:
        for record in records:
            handle.write(json.dumps(record, ensure_ascii=False) + "\n")

    temporary.replace(path)
    return changed_records, changed_paths


instruction_root = PROJECT_ROOT / "data/processed/visa_instructions"

for split_name in ("train", "validation", "test"):
    split_path = instruction_root / f"{split_name}.jsonl"
    changed_records, changed_paths = normalize_instruction_jsonl(split_path)
    print(
        f"{split_name}: changed_records={changed_records}, "
        f"changed_paths={changed_paths}"
    )

# Verify one path resolves correctly before loading the model.
train_path = instruction_root / "train.jsonl"
with train_path.open("r", encoding="utf-8") as handle:
    first_record = json.loads(next(handle))

image_reference = next(
    item["image"]
    for message in first_record["messages"]
    for item in message.get("content", [])
    if item.get("type") == "image"
)
resolved_image = PROJECT_ROOT / image_reference

print("Example image reference:", image_reference)
print("Resolved image:", resolved_image)
print("Image exists:", resolved_image.is_file())

assert image_reference.startswith("data/raw/visa/")
assert resolved_image.is_file()


train: changed_records=37005, changed_paths=37005
validation: changed_records=7926, changed_paths=7926
test: changed_records=7932, changed_paths=7932
Example image reference: data/raw/visa/candle/Data/Images/Anomaly/003.JPG
Resolved image: /content/visionassist-industrial-visual-inspection/data/raw/visa/candle/Data/Images/Anomaly/003.JPG
Image exists: True


In [8]:
#@title 8. Configure OOM-safe A100 profile and persistent checkpoints
import yaml

cfg = yaml.safe_load(OVERFIT_CONFIG.read_text(encoding="utf-8"))

# Memory-bounded QLoRA settings for Qwen2.5-VL-3B on an A100 40 GB.
cfg["attention_implementation"] = "sdpa"
cfg["lora"]["rank"] = 8
cfg["lora"]["alpha"] = 16
cfg["lora"]["target_suffixes"] = ["q_proj", "v_proj", "o_proj"]
cfg["lora"]["train_multimodal_projector"] = False

cfg["data"]["max_sequence_length"] = 2048
cfg["data"]["image_min_pixels"] = 100352
cfg["data"]["image_max_pixels"] = 200704

cfg["training"]["per_device_train_batch_size"] = 1
cfg["training"]["per_device_eval_batch_size"] = 1
cfg["training"]["gradient_accumulation_steps"] = 4
cfg["training"]["gradient_checkpointing"] = True
cfg["training"]["save_total_limit"] = 3

cfg["checkpoints"]["resume"] = "latest"
cfg["checkpoints"]["keep_latest"] = 2
cfg["checkpoints"]["keep_best"] = 1
cfg["checkpoints"]["persistent_output_dir"] = str(DRIVE_CHECKPOINT_ROOT)
cfg["checkpoints"]["sync_every_save"] = True

OVERFIT_CONFIG.write_text(
    yaml.safe_dump(cfg, sort_keys=False),
    encoding="utf-8",
)

print(yaml.safe_dump(cfg, sort_keys=False))

assert cfg["data"]["max_sequence_length"] == 2048
assert cfg["data"]["image_max_pixels"] == 200704
assert cfg["lora"]["rank"] == 8
assert cfg["lora"]["target_suffixes"] == ["q_proj", "v_proj", "o_proj"]


run_id: qwen25vl3b_qlora_overfit_v1
model_id: Qwen/Qwen2.5-VL-3B-Instruct
model_revision: null
processor_revision: null
output_dir: outputs/training/qwen25vl3b_qlora_overfit_v1
seed: 42
trust_remote_code: false
attention_implementation: sdpa
hardware_profile: auto
quantization:
  enabled: true
  load_in_4bit: true
  quant_type: nf4
  double_quant: true
  compute_dtype: auto
lora:
  rank: 8
  alpha: 16
  dropout: 0.05
  bias: none
  target_suffixes:
  - q_proj
  - v_proj
  - o_proj
  train_multimodal_projector: false
data:
  train_path: data/processed/visa_instructions/train.jsonl
  validation_path: data/processed/visa_instructions/validation.jsonl
  max_sequence_length: 2048
  image_min_pixels: 100352
  image_max_pixels: 200704
  train_limit: 32
  validation_limit: 32
  subset_seed: 42
training:
  max_steps: 100
  num_train_epochs: 1.0
  learning_rate: 0.0002
  per_device_train_batch_size: 1
  per_device_eval_batch_size: 1
  gradient_accumulation_steps: 4
  gradient_checkpointing: true

In [9]:
#@title 9. Inspect Phase 8 environment
import subprocess
subprocess.run(["uv","run","visionassist","training-environment","--config",str(OVERFIT_CONFIG)], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['uv', 'run', 'visionassist', 'training-environment', '--config', '/content/visionassist-industrial-visual-inspection/configs/training/qwen25vl3b_qlora_overfit.yaml'], returncode=0)

In [10]:
#@title 10. Run local unit tests before GPU use
import subprocess
subprocess.run(["uv","run","pytest","tests/test_phase8_training.py"], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['uv', 'run', 'pytest', 'tests/test_phase8_training.py'], returncode=0)

In [11]:
#@title 11. One-batch forward-pass smoke test with diagnostics
import gc
import subprocess
import torch

gc.collect()
torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
free_bytes, total_bytes = torch.cuda.mem_get_info()
print(f"Free VRAM before smoke test: {free_bytes / 1024**3:.2f} GiB")

command = [
    "uv",
    "run",
    "visionassist",
    "training-smoke-test",
    "--config",
    str(OVERFIT_CONFIG),
]

result = subprocess.run(
    command,
    cwd=PROJECT_ROOT,
    text=True,
    capture_output=True,
)

print("Return code:", result.returncode)
print("\n========== STDOUT ==========")
print(result.stdout or "<empty>")
print("\n========== STDERR ==========")
print(result.stderr or "<empty>")

report_paths = [
    PROJECT_ROOT
    / "outputs/training/qwen25vl3b_qlora_overfit_v1/one_batch_smoke_test.json",
    PROJECT_ROOT
    / "outputs/training/qwen25vl3b_qlora_overfit_v1/run_manifest.json",
    PROJECT_ROOT
    / "outputs/training/qwen25vl3b_qlora_overfit_v1/environment.json",
]

for report_path in report_paths:
    print(f"\n========== {report_path.relative_to(PROJECT_ROOT)} ==========")
    if report_path.is_file():
        print(report_path.read_text(encoding="utf-8")[:20_000])
    else:
        print("<not created>")

if result.returncode != 0:
    raise RuntimeError(
        "Training smoke test failed. Review STDERR above for the actual cause."
    )


Free VRAM before smoke test: 39.08 GiB
Return code: 0

========== STDOUT ==========
Phase 8 one-batch smoke test passed.
Loss: 3.0354368686676025
Batch shape: [1, 280]
Trainable parameters: 3022848


========== STDERR ==========

Fetching 2 files: 100%|██████████| 2/2 [00:22<00:00, 11.29s/it]

Loading weights: 100%|██████████| 824/824 [00:02<00:00, 339.46it/s]
/content/visionassist-industrial-visual-inspection/.venv/lib/python3.12/site-packages/torch/_dynamo/eval_frame.py:1446: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. Starting in PyTorch 2.9, calling checkpoint without use_reentrant will raise an exception. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/content/visionassist-industrial-visual-inspection/.venv/lib/python3.12/site-packages/bitsandbytes/b

## 32-example overfit run

This run saves every 10 steps. Local `/content` retains at most 3 checkpoints.
Drive retains the newest two plus the best checkpoint. Run the same command after
a disconnect; `--resume latest` restores the newest persistent checkpoint.

In [14]:
from pathlib import Path

train_file = (
    PROJECT_ROOT
    / "src/visionassist/training/train.py"
)

text = train_file.read_text(encoding="utf-8")

old = "        save_safetensors=True,\n"

if old in text:
    text = text.replace(old, "")
    train_file.write_text(text, encoding="utf-8")
    print("Removed unsupported save_safetensors argument.")
else:
    print("save_safetensors argument was not found or was already removed.")

Removed unsupported save_safetensors argument.


In [15]:
text = train_file.read_text(encoding="utf-8")

assert "save_safetensors=" not in text
print("TrainingArguments compatibility fix verified.")

TrainingArguments compatibility fix verified.


In [16]:
#@title 12. Train or resume the overfit run
import subprocess
subprocess.run([
    "uv","run","visionassist","train-qlora",
    "--config",str(OVERFIT_CONFIG),
    "--resume","latest",
], cwd=PROJECT_ROOT, check=True)

CompletedProcess(args=['uv', 'run', 'visionassist', 'train-qlora', '--config', '/content/visionassist-industrial-visual-inspection/configs/training/qwen25vl3b_qlora_overfit.yaml', '--resume', 'latest'], returncode=0)

In [17]:
#@title 13. Inspect retained Drive checkpoints
from pathlib import Path
checkpoints = sorted(DRIVE_CHECKPOINT_ROOT.glob("checkpoint-*"), key=lambda p:int(p.name.split('-')[-1]))
print("Retained checkpoints:")
for path in checkpoints: print(path.name)
assert len(checkpoints) <= 3

Retained checkpoints:
checkpoint-50
checkpoint-90
checkpoint-100


In [18]:
#@title 14. Sync final adapter and lightweight reports to Drive
import shutil
local_run = PROJECT_ROOT/"outputs/training"/OVERFIT_RUN_ID
drive_run = DRIVE_ROOT/"outputs/training"/OVERFIT_RUN_ID
drive_run.mkdir(parents=True, exist_ok=True)
for name in ["resolved_config.yaml","run_manifest.json","dataset_manifest.json","environment.json","trainable_parameters.json","one_batch_smoke_test.json"]:
    source=local_run/name
    if source.is_file(): shutil.copy2(source, drive_run/name)
if (local_run/"final_adapter").is_dir():
    shutil.copytree(local_run/"final_adapter", drive_run/"final_adapter", dirs_exist_ok=True)
print("Synced to", drive_run)

Synced to /content/drive/MyDrive/visionassist/outputs/training/qwen25vl3b_qlora_overfit_v1


## Inspect the final trainer state and confirm the best checkpoint

In [19]:
import json
from pathlib import Path

run_dir = (
    PROJECT_ROOT
    / "outputs/training"
    / "qwen25vl3b_qlora_overfit_v1"
)

candidate_state_files = [
    run_dir / "trainer_state.json",
    run_dir / "checkpoint-100" / "trainer_state.json",
]

state_path = next(
    (path for path in candidate_state_files if path.is_file()),
    None,
)

if state_path is None:
    raise FileNotFoundError("No trainer_state.json found.")

state = json.loads(state_path.read_text(encoding="utf-8"))

print("Trainer state:", state_path)
print("Global step:", state.get("global_step"))
print("Best checkpoint:", state.get("best_model_checkpoint"))
print("Best metric:", state.get("best_metric"))

print("\nRecent log history:")
for entry in state.get("log_history", [])[-10:]:
    print(entry)

Trainer state: /content/visionassist-industrial-visual-inspection/outputs/training/qwen25vl3b_qlora_overfit_v1/trainer_state.json
Global step: 100
Best checkpoint: outputs/training/qwen25vl3b_qlora_overfit_v1/checkpoint-50
Best metric: 1.2456169128417969

Recent log history:
{'epoch': 11.0, 'grad_norm': 0.23833155632019043, 'learning_rate': 8.733488479845997e-06, 'loss': 0.00928587093949318, 'step': 88}
{'epoch': 11.25, 'grad_norm': 1.9161303043365479, 'learning_rate': 6.2793294993656494e-06, 'loss': 0.13028085231781006, 'step': 90}
{'epoch': 11.25, 'eval_loss': 1.4156767129898071, 'eval_runtime': 10.0461, 'eval_samples_per_second': 3.185, 'eval_steps_per_second': 3.185, 'step': 90}
{'epoch': 11.5, 'grad_norm': 0.6524654626846313, 'learning_rate': 4.2182675812012965e-06, 'loss': 0.04044613242149353, 'step': 92}
{'epoch': 11.75, 'grad_norm': 0.9969503283500671, 'learning_rate': 2.5589475353073988e-06, 'loss': 0.05320902541279793, 'step': 94}
{'epoch': 12.0, 'grad_norm': 0.06748455017805

## verify the final adapter exists

In [20]:
final_adapter = run_dir / "final_adapter"

print("Final adapter exists:", final_adapter.is_dir())

if final_adapter.is_dir():
    for path in sorted(final_adapter.iterdir()):
        print(path.name)

Final adapter exists: True
README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
processor_config.json
tokenizer.json
tokenizer_config.json


## Resume after a disconnect

Run cells 1–9 again, including the path-normalization cell, then run cell 12. The command uses `resume: latest`, checks
both local and Drive checkpoint roots, copies the newest Drive checkpoint back
to `/content`, and resumes optimizer/scheduler/trainer state.

The best checkpoint is selected by minimum `eval_loss`; the final Trainer state
records `best_model_checkpoint` and `best_metric`.